In [1]:

import numpy as np

# =====================================
# 1. TRUSS STRUCTURAL MODEL
# =====================================

nodes = np.array([
    [0, 0],
    [1, 0],
    [2, 0],
    [1, 1]
], dtype=float)

members = np.array([
    [0, 1],
    [1, 2],
    [0, 3],
    [1, 3],
    [2, 3]
])

E = 200e9
rho = 7850
stress_limit = 250e6
displacement_limit = 0.01

A_min = 1e-4
A_max = 5e-3

# Fixed supports: node 0 and node 2
fixed_dofs = [0, 1, 4, 5]

# External load
F = np.zeros(2 * len(nodes))
F[7] = -10000


# =====================================
# 2. FINITE ELEMENT ANALYSIS
# =====================================

def analyze(A):

    ndof = 2 * len(nodes)
    K = np.zeros((ndof, ndof))
    lengths = []
    directions = []

    for i, (n1, n2) in enumerate(members):

        dx, dy = nodes[n2] - nodes[n1]
        L = np.hypot(dx, dy)

        c = dx / L
        s = dy / L

        k = E * A[i] / L * np.array([
            [c*c, c*s, -c*c, -c*s],
            [c*s, s*s, -c*s, -s*s],
            [-c*c, -c*s, c*c, c*s],
            [-c*s, -s*s, c*s, s*s]
        ])

        dofs = [
            2*n1, 2*n1+1,
            2*n2, 2*n2+1
        ]

        K[np.ix_(dofs, dofs)] += k

        lengths.append(L)
        directions.append((c, s))

    free = np.setdiff1d(
        np.arange(ndof), fixed_dofs
    )

    try:
        U = np.zeros(ndof)

        U[free] = np.linalg.solve(
            K[np.ix_(free, free)], F[free]
        )

        stress = []

        for i, (n1, n2) in enumerate(members):

            c, s = directions[i]
            L = lengths[i]

            dofs = [
                2*n1, 2*n1+1,
                2*n2, 2*n2+1
            ]

            u = U[dofs]

            strain = np.dot(
                [-c, -s, c, s], u
            ) / L

            stress.append(E * strain)

        weight = rho * np.sum(A * lengths)

        return (
            weight,
            np.max(np.abs(stress)),
            np.max(np.abs(U)),
            np.array(stress)
        )

    except np.linalg.LinAlgError:
        return 1e12, 1e12, 1e12, np.zeros(len(members))


# =====================================
# 3. FITNESS FUNCTION
# =====================================

def fitness(A):

    weight, stress, disp, _ = analyze(A)

    penalty = (
        max(0, stress / stress_limit - 1)**2
        + max(0, disp / displacement_limit - 1)**2
    )

    return weight + 1e6 * penalty


# =====================================
# 4. CELLULAR AUTOMATA OPTIMIZATION
# =====================================

def cellular_optimization():

    np.random.seed(42)

    rows, cols = 4, 4

    population = np.random.uniform(
        A_min, A_max, (rows, cols, len(members))
    )

    generations = 100

    best_design = None
    best_fitness = float("inf")

    for generation in range(generations):

        new_population = population.copy()

        for i in range(rows):
            for j in range(cols):

                current = population[i, j]

                # Moore neighbourhood
                neighbours = []

                for di in [-1, 0, 1]:
                    for dj in [-1, 0, 1]:

                        ni = (i + di) % rows
                        nj = (j + dj) % cols

                        if (ni, nj) != (i, j):
                            neighbours.append(
                                population[ni, nj]
                            )

                neighbour_scores = [
                    fitness(x) for x in neighbours
                ]

                best_neighbour = neighbours[
                    np.argmin(neighbour_scores)
                ]

                # Local cellular update
                child = (
                    0.7 * current
                    + 0.3 * best_neighbour
                )

                # Small mutation
                mutation = np.random.normal(
                    0, 0.0001, len(members)
                )

                child += mutation

                child = np.clip(
                    child, A_min, A_max
                )

                # Accept only improved designs
                if fitness(child) < fitness(current):
                    new_population[i, j] = child

        population = new_population

        scores = np.array([
            fitness(population[i, j])
            for i in range(rows)
            for j in range(cols)
        ])

        idx = np.argmin(scores)

        if scores[idx] < best_fitness:

            best_fitness = scores[idx]

            best_design = population.reshape(
                -1, len(members)
            )[idx].copy()

        print(
            f"Generation {generation+1}: "
            f"Best fitness = {best_fitness:.4f}"
        )

    return best_design


# =====================================
# 5. DISPLAY RESULTS
# =====================================

if __name__ == "__main__":

    best_A = cellular_optimization()

    weight, stress, disp, member_stress = analyze(
        best_A
    )

    print("\nOPTIMIZED STRUCTURAL DESIGN")

    print("Member areas (m2):", best_A)

    print("Minimum weight (kg):", weight)

    print("Maximum stress (MPa):", stress / 1e6)

    print("Maximum displacement (m):", disp)

    print("Stress constraint satisfied:",
          stress <= stress_limit)

    print("Displacement constraint satisfied:",
          disp <= displacement_limit)


Generation 1: Best fitness = 73.5370
Generation 2: Best fitness = 73.0589
Generation 3: Best fitness = 70.7476
Generation 4: Best fitness = 70.6673
Generation 5: Best fitness = 70.6673
Generation 6: Best fitness = 69.6727
Generation 7: Best fitness = 68.5192
Generation 8: Best fitness = 65.0880
Generation 9: Best fitness = 65.0880
Generation 10: Best fitness = 62.3364
Generation 11: Best fitness = 62.3364
Generation 12: Best fitness = 61.0063
Generation 13: Best fitness = 61.0063
Generation 14: Best fitness = 58.1695
Generation 15: Best fitness = 58.1695
Generation 16: Best fitness = 56.6144
Generation 17: Best fitness = 55.2995
Generation 18: Best fitness = 54.1810
Generation 19: Best fitness = 54.1810
Generation 20: Best fitness = 51.8349
Generation 21: Best fitness = 51.8349
Generation 22: Best fitness = 49.9634
Generation 23: Best fitness = 49.4705
Generation 24: Best fitness = 49.4705
Generation 25: Best fitness = 48.1193
Generation 26: Best fitness = 45.1886
Generation 27: Best f